In [29]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    roc_curve
)
import matplotlib.pyplot as plt
import joblib

In [30]:
df = pd.read_csv("cleaned_data.csv")
df = df.drop(columns=["dteday", "instant", "casual", "registered"], errors="ignore")


In [31]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y_clf,
    test_size=0.2,
    random_state=42,
    stratify=y_clf
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

In [32]:

# Encode categorical columns
nominal_cols = ["season_label", "mnth", "weekday"]
df = pd.get_dummies(df, columns=nominal_cols, drop_first=True)

X = df.drop("cnt", axis=1)
y_regr = df["cnt"]
y_clf = (y_regr > y_regr.median()).astype(int)

In [33]:
dt1 = DecisionTreeClassifier(random_state=42)
dt1.fit(X_train, y_train)
print("Train Acc:", accuracy_score(y_train, dt1.predict(X_train)))
print("Test Acc:", accuracy_score(y_test, dt1.predict(X_test)))

Train Acc: 1.0
Test Acc: 0.9115646258503401


In [34]:
dt2 = DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=42)
dt2.fit(X_train, y_train)
print("Train Acc:", accuracy_score(y_train, dt2.predict(X_train)))
print("Test Acc:", accuracy_score(y_test, dt2.predict(X_test)))

Train Acc: 0.9143835616438356
Test Acc: 0.8707482993197279


In [35]:
gini = DecisionTreeClassifier(max_depth=5, criterion="gini", random_state=42)
entropy = DecisionTreeClassifier(max_depth=5, criterion="entropy", random_state=42)
gini.fit(X_train, y_train)
entropy.fit(X_train, y_train)
print("Gini Test:", accuracy_score(y_test, gini.predict(X_test)))
print("Entropy Test:", accuracy_score(y_test, entropy.predict(X_test)))

Gini Test: 0.9047619047619048
Entropy Test: 0.9047619047619048


In [36]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)
rf.fit(X_train, y_train)
print("Train Acc:", accuracy_score(y_train, rf.predict(X_train)))
print("Test Acc:", accuracy_score(y_test, rf.predict(X_test)))
print("AUC:", roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1]))

Train Acc: 0.9897260273972602
Test Acc: 0.9047619047619048
AUC: 0.9716771566086634


In [37]:
importance = pd.Series(rf.feature_importances_, index=X.columns)
print(importance.sort_values(ascending=False).head(5))

atemp        0.195694
yr           0.177179
temp         0.171493
hum          0.086013
windspeed    0.074244
dtype: float64


In [38]:
gbc = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
gbc.fit(X_train, y_train)
print("Test Acc:", accuracy_score(y_test, gbc.predict(X_test)))
print("AUC:", roc_auc_score(y_test, gbc.predict_proba(X_test)[:, 1]))

Test Acc: 0.9183673469387755
AUC: 0.9776008885597927


In [39]:
low_feat = importance.sort_values().head(5).index

X_train_df = pd.DataFrame(X_train, columns=X.columns)
X_test_df = pd.DataFrame(X_test, columns=X.columns)

X_train_red = X_train_df.drop(columns=low_feat)
X_test_red = X_test_df.drop(columns=low_feat)

rf2 = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf2.fit(X_train_red, y_train)
print("Full AUC:", roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1]))
print("Reduced AUC:", roc_auc_score(y_test, rf2.predict_proba(X_test_red)[:, 1]))

Full AUC: 0.9716771566086634
Reduced AUC: 0.9744539059607552


In [40]:
models = {
    "Log_Reg": LogisticRegression(max_iter=5000),
    "Tree": DecisionTreeClassifier(max_depth=5),
    "RF": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    "GB": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3)
}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, model in models.items():
    scores = cross_val_score(model, X, y_clf, cv=skf, scoring="roc_auc")
    print(name, scores.mean(), scores.std())

Log_Reg 0.9463947903617635 0.008913131415458181
Tree 0.9185002510485717 0.017521621936653096
RF 0.963833790630563 0.005316572265813722
GB 0.9597165940569956 0.009843483783018811


In [41]:
pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(random_state=42))
])

param_grid = {
    "model__n_estimators": [50, 100, 200],
    "model__max_depth": [5, 10, None],
    "model__min_samples_leaf": [1, 5]
}

grid = GridSearchCV(
    pipe,
    param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring="roc_auc",
    n_jobs=-1
)

grid.fit(X_train_raw, y_train)
print(grid.best_params_)
print(grid.best_score_)

{'model__max_depth': None, 'model__min_samples_leaf': 5, 'model__n_estimators': 50}
0.959378463894879


In [42]:
frac = [0.2, 0.4, 0.6, 0.8, 1.0]

best_model = grid.best_estimator_

for f in frac:
    size = int(f * len(X_train_raw))

    X_sub = X_train_raw.iloc[:size]
    y_sub = y_train.iloc[:size]

    best_model.fit(X_sub, y_sub)

    train_auc = roc_auc_score(y_sub, best_model.predict_proba(X_sub)[:, 1])
    test_auc = roc_auc_score(y_test, best_model.predict_proba(X_test_raw)[:, 1])

    print(f, train_auc, test_auc)

0.2 0.9802867383512545 0.947611995557201
0.4 0.9835920177383592 0.944094779711218
0.6 0.9855631042592109 0.9642724916697519
0.8 0.987343165309267 0.9742687893372824
1.0 0.9897377556764871 0.9713069233617179


In [43]:
joblib.dump(grid.best_estimator_, "best_model.pkl")

['best_model.pkl']

In [44]:
model = joblib.load("best_model.pkl")
print(model.predict(X_test_raw[:2]))  # change from: X_test[:2]

[1 1]
